## `ApplicationContext` events

Beans can broadcast events to each other through the container. This decouples a producer from the consumers — the producer doesn't know who, if anyone, listens.

```java
// the event — a record is ideal
public record PaymentProcessed(long paymentId, long amount, Instant at) {}

// the publisher — inject ApplicationEventPublisher
@Service
public class PaymentService {
    private final ApplicationEventPublisher events;
    public PaymentService(ApplicationEventPublisher events) { this.events = events; }

    public void process(Payment p) {
        // ... work ...
        events.publishEvent(new PaymentProcessed(p.id(), p.amount(), Instant.now()));
    }
}

// any number of listeners — discovered via reflection
@Component
public class AuditListener {
    @EventListener
    public void onPayment(PaymentProcessed event) { /* log to audit table */ }
}
```

Events are synchronous by default — `publishEvent` blocks until every listener returns. Add `@Async` to a listener (plus `@EnableAsync`) for asynchronous delivery. Spring also ships built-in events like `ContextRefreshedEvent` and `ContextClosedEvent` for startup/shutdown hooks.
